In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, b4eac297-bb84-4aa3-b858-dd74d873b432, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, b4eac297-bb84-4aa3-b858-dd74d873b432, 4, Finished, Available, Finished, False)

16 projects found


In [3]:
all_snapshots = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling snapshots for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.0/budget_view_snapshots",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code} for {project_name}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_snapshots.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total snapshots found: {len(all_snapshots)}")

StatementMeta(, b4eac297-bb84-4aa3-b858-dd74d873b432, 5, Finished, Available, Finished, False)

Pulling snapshots for: 1100 Fulton Street
Pulling snapshots for: 11 ESSEX ST
Pulling snapshots for: 337A & 337B West Broadway Rehabilitaion Work
Pulling snapshots for: 360 Lexington 8th & 20th Floor
Pulling snapshots for: 549 Munroe Av
Pulling snapshots for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling snapshots for: Boys & Girls Club
Pulling snapshots for: EMBANKMENT PHASE II
Pulling snapshots for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling snapshots for: Lillipvt 45 Renwick St
Pulling snapshots for: PCNA 711 11TH AVE
Pulling snapshots for: Sandbox Test Project
Pulling snapshots for: Standard Project Template
Pulling snapshots for: SYMRISE - 15th & 16th Flr
Pulling snapshots for: TEST - ABM SUBORDINATE
Pulling snapshots for: VOCO HOTEL TSQ
Done! Total snapshots found: 62


In [4]:
all_detail_rows = []

for snapshot in all_snapshots:
    snapshot_id = snapshot["id"]
    project_id = snapshot["project_id"]
    project_name = snapshot["project_name"]
    snapshot_name = snapshot.get("name", "")
    snapshot_type = snapshot.get("snapshot_type", "")
    created_at = snapshot.get("created_at", "")

    print(f"Pulling detail rows for snapshot: {snapshot_name} | {project_name}")

    page = 1
    while True:
        response = requests.get(
            f"https://api.procore.com/rest/v1.0/budget_view_snapshots/{snapshot_id}/detail_rows",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["snapshot_id"] = snapshot_id
            row["snapshot_name"] = snapshot_name
            row["snapshot_type"] = snapshot_type
            row["snapshot_created_at"] = created_at
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_detail_rows.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total detail rows: {len(all_detail_rows)}")

StatementMeta(, b4eac297-bb84-4aa3-b858-dd74d873b432, 6, Finished, Available, Finished, False)

Pulling detail rows for snapshot: MARCH MTM - 4.10.26 | 11 ESSEX ST
Pulling detail rows for snapshot: FEB BUDGET - 3.13.26 | 11 ESSEX ST
Pulling detail rows for snapshot: FEB MTM - 3.13.26 | 11 ESSEX ST
Pulling detail rows for snapshot: JAN MTM - 2.19.26 | 11 ESSEX ST
Pulling detail rows for snapshot: JAN MTM - 2.16.26 | 11 ESSEX ST
Pulling detail rows for snapshot: DECEMBER DRAFT | 11 ESSEX ST
Pulling detail rows for snapshot: NOVEMEBER Budget Snapshot  | 11 ESSEX ST
Pulling detail rows for snapshot: NOVEMEBER Snapshot  | 11 ESSEX ST
Pulling detail rows for snapshot: OCTOBER 2025 | 11 ESSEX ST
Pulling detail rows for snapshot: Test- November  | 11 ESSEX ST
Pulling detail rows for snapshot: OCT '25 COST REPORT | 11 ESSEX ST
Pulling detail rows for snapshot: MARCH MTM - 4.10.26 | 360 Lexington 8th & 20th Floor
Pulling detail rows for snapshot: MTM 4.10 | 360 Lexington 8th & 20th Floor
Pulling detail rows for snapshot: BUDGET SNAPSHOT 4.10 | 360 Lexington 8th & 20th Floor
Pulling detail 

In [5]:
import pandas as pd
import re

clean_rows = []
for row in all_detail_rows:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("procore_budget_snapshots_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, b4eac297-bb84-4aa3-b858-dd74d873b432, 7, Finished, Available, Finished, False)

UnsupportedOperationException: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.